In [1]:
from openrouter import OpenRouter
from prompt_utils import *
from pprint import pprint

import json
import random
import os
import requests

In [ ]:
# Configurations first
model = 'deepseek'
model_name = "deepseek/deepseek-v4.1-flash"

# Load the prompts
system_prompt = stance_emotion_system_prompt()
user_prompt = stance_emotion_user_prompt()

# Load the LLM engine
api_key = ""
open_router_chat = OpenRouter(api_key=api_key)

In [3]:
# OpenRouter related configs
batch_url = "https://openrouter.ai/api/v1/batches"
batch_model = "deepseek/deepseek-v4.1-flash:batch"

headers={
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

batch_send_endpoint = "/v1/chat/completions"

In [4]:
# Load existing processed comments
processed_file = f'{model}_emotions_stance_processed'
processed_data = []

# Check if path exists
if os.path.exists(f'{processed_file}.json'):
    with open(f'{processed_file}.json', "r", encoding="utf-8") as f:
        processed_data = json.load(f)

print(f"Size of processed data: {len(processed_data)}")

Size of processed data: 0


In [5]:
# Load the json file
filename = "all_comments.json"

new_data = []

with open(filename, "r", encoding="utf-8") as f:
    new_data = json.load(f)

print(f"Loaded: {len(new_data)} comments")

Loaded: 1734 comments


In [6]:
# Perform subtraction

# Get the processed IDs
processed_ids = {item['row_id'] for item in processed_data}

# Remove
new_data = [item for item in new_data if item['row_id'] not in processed_ids]

print(f"After subtraction: {len(new_data)}")

# Adjusting nan values


After subtraction: 1734


In [7]:
# Prepare the user prompts

# Function for preparing the body
def get_body_request(record_id, system_prompt, selected_prompt):
    return {
        "custom_id": record_id,
        "body": {
          "messages": [
              {
                  "role": "system",
                  "content": system_prompt
              },
              {
                  "role":"user",
                  "content": selected_prompt
              }
          ]
        }
      }

batch_size = 100
batched_prompts = []

current_batch = []
current_data = {}

for record in new_data:

    # Handle NaN
    if isinstance(record['parent_text'], float):
        record["parent_text"] = ""

    selected_prompt = user_prompt.format(
        parent_comment = record['parent_text'],
        post_comment = record['post_comment']
    )

    # Prepare the request body
    record_id = record['post_id']
    request_body = get_body_request(record_id, system_prompt, selected_prompt)

    # For reference purposes later on
    current_data[record_id] = record

    # Add to the batch
    current_batch.append(request_body)

    # Check if size reached
    # If it is, reset the current batch and add existing batch to batched prompts
    if len(current_batch) == batch_size:
        batched_prompts.append(current_batch)
        current_batch = []


# For leftovers
if current_batch:
    batched_prompts.append(current_batch)

In [8]:

batch_list = []

# Loop through one batch at a time
for batch in batched_prompts:

    # Send for prompting
    response = requests.post(
        url=batch_url,
        headers=headers,

        data=json.dumps({
            "endpoint": batch_send_endpoint,
            "model": batch_model,
            "requests": batch
        })
    )

    # Save the batch ID
    response = response.json()
    batch_list.append(response['id'])


In [39]:
# Get list of batches

res = open_router_chat.batch.list()
print(res)

batch_list = [
    batch.id
    for batch in res.result.data
]

print(len(batch_list))

next=<function Batch.list.<locals>.next_func at 0x00000269B902E560> result=BatchListResponse(data=[BatchListItem(completion_window='24h', created_at=1791114432, endpoint='/v1/chat/completions', error=None, finalized_at=1791116212, id='batch-1791114432-32UT8haorbGvMqOMOA10', model='deepseek/deepseek-v4.1-flash-20260910', object='batch', request_counts=BatchListItemRequestCounts(completed=34, failed=0, total=34), results=None, status='completed', usage=BatchListItemUsage(completion_tokens=29557, prompt_tokens=23312, total_tokens=52869, cache_creation=Unset(), completion_tokens_details=Unset(), cost=0.010623079, cost_details=Unset(), is_byok=False, iterations=Unset(), prompt_tokens_details=Unset(), server_tool_use=Unset(), service_tier=Unset(), speed=Unset())), BatchListItem(completion_window='24h', created_at=1791114431, endpoint='/v1/chat/completions', error=None, finalized_at=1791116211, id='batch-1791114431-tfhbT9wiPZbAKq2t3xxb', model='deepseek/deepseek-v4.1-flash-20260910', object='

completed
{'total': 34, 'completed': 34, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}


completed
{'total': 100, 'completed': 100, 'failed': 0}




In [41]:
print(len(batch_list))
pprint(batch_list)

18
['batch-1791114432-32UT8haorbGvMqOMOA10',
 'batch-1791114431-tfhbT9wiPZbAKq2t3xxb',
 'batch-1791114429-bXvXVyzXQIllBMAD6ktF',
 'batch-1791114427-9ON3vABTiJVCJ7xtO4hv',
 'batch-1791114425-Q47j0RuRS3XefxQd6J8Z',
 'batch-1791114424-8OeYSzh8sfWpmhGJg1fM',
 'batch-1791114422-uBoMOmdUlHIWbDNDllVS',
 'batch-1791114420-0ptDp1uLufD81nVNY9r3',
 'batch-1791114419-a4C26o9hNmq9Bt3uKwnK',
 'batch-1791114417-ZEKqQHt4ivkSt19btnpW',
 'batch-1791114416-I4C0ekMm7ULsYx7nVOmK',
 'batch-1791114414-1vQLoCiNUNmVe3o30KUB',
 'batch-1791114412-vktTWhlEu2hu83NY6mb1',
 'batch-1791114410-hlXI9DJZrU8q2jfuEWqL',
 'batch-1791114408-k6SPKFLuTpW8a4ncpuUs',
 'batch-1791114405-1szgV4aOakZvTmmtYuwz',
 'batch-1791114403-pIOVLTqbvoMx7lwjTSif',
 'batch-1791114401-XlXbttkDhCFvnMCiXNeh']


In [54]:

# Go through the list. Stop only when the list is empty
current_index = 0

while batch_list:

    curr_batch_id = batch_list[current_index]

    # Check the status
    url = f"https://openrouter.ai/api/v1/batches/{curr_batch_id}"
    response = requests.get(url, headers=headers)
    response = response.json()

    # If still in progress, move to the next one
    if response['status'] == 'in_progress':

        current_index = current_index + 1 if current_index < len(batch_list) else 0
        continue

    # Parse the processed results one by one
    for result in response['results']:

        # Go by the given row ID
        post_id = result['custom_id']

        # Convert to JSON
        llm_output = result['response']['body']['choices'][0]['message']['content']

        try:
            llm_output = json.loads(llm_output)
        except (json.JSONDecodeError, TypeError) as e:
            pprint(llm_output)
            continue

        # Combine the dictionary of the given row ID
        current_data[post_id] |= llm_output

        reasoning_list.append({
            'post_id': post_id,
            'reasoning': result['response']['body']['choices'][0]['message']['reasoning']
        })


    # Save the output to JSON file
    with open(f"{processed_file}.json", "w", encoding="utf-8") as f:
        values = list(current_data.values())
        json.dump(values, f, ensure_ascii=False, indent=2)

    with open(f"deepseek_reasoning.json", "w", encoding="utf-8") as f:
        json.dump(reasoning_list, f, ensure_ascii=False, indent=2)

    # Delete the batch
    batch_list.remove(curr_batch_id)

    # Finished one, so we start from 0 again
    current_index = 0

('## Coding\n'
 '\n'
 '**Target comment:** *"We addicted you so much to your phone that we know '
 'you\'ve gotten dumber. So here use our AI to make you smarter."*\n'
 '\n'
 '```json\n'
 '{\n'
 '  "sarcasm_detected": true,\n'
 '  "irony_detected": true,\n'
 '  "stance_label": "against",\n'
 '  "relation_label": "not_applicable",\n'
 '  "emotion": "Disgust"\n'
 '}\n'
 '```\n'
 '\n'
 '### Why\n'
 '\n'
 '- **sarcasm_detected / irony_detected — true.** The comment is a mocking '
 "paraphrase of the ad's implicit pitch. It says the *opposite* of a sincere "
 'endorsement ("here, use our AI") while the framing ("we addicted you… '
 'you\'ve gotten dumber") makes the intent clearly critical. Nothing about it '
 'is a genuine recommendation.\n'
 '\n'
 '- **stance_label — against.** The intended meaning targets *the use of AI '
 'here*, not the product or ad quality in general: the AI is framed as a '
 'cynical remedy for a problem the company itself created (phone addiction → '
 '"dumber"). T

In [57]:
# Delete batches
res = open_router_chat.batch.list()
print(res)

for batch in res.result.data:
    url = f"https://openrouter.ai/api/v1/batches/{batch.id}"

    response = requests.delete(url, headers=headers)
    print(f"{batch.id} deleted!")

next=<function Batch.list.<locals>.next_func at 0x00000269BA422400> result=BatchListResponse(data=[BatchListItem(completion_window='24h', created_at=1791114432, endpoint='/v1/chat/completions', error=None, finalized_at=1791116212, id='batch-1791114432-32UT8haorbGvMqOMOA10', model='deepseek/deepseek-v4.1-flash-20260910', object='batch', request_counts=BatchListItemRequestCounts(completed=34, failed=0, total=34), results=None, status='completed', usage=BatchListItemUsage(completion_tokens=29557, prompt_tokens=23312, total_tokens=52869, cache_creation=Unset(), completion_tokens_details=Unset(), cost=0.010623079, cost_details=Unset(), is_byok=False, iterations=Unset(), prompt_tokens_details=Unset(), server_tool_use=Unset(), service_tier=Unset(), speed=Unset())), BatchListItem(completion_window='24h', created_at=1791114431, endpoint='/v1/chat/completions', error=None, finalized_at=1791116211, id='batch-1791114431-tfhbT9wiPZbAKq2t3xxb', model='deepseek/deepseek-v4.1-flash-20260910', object='